In [1]:
#pip install opencv-python mediapipe numpy

In [2]:
#pip install --upgrade mediapipe

In [3]:
import threading
import asyncio
import edge_tts
import pygame
import os
import time

pygame 2.6.1 (SDL 2.28.4, Python 3.13.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


c:\Users\theda\anaconda3\envs\HAR\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [1]:
import threading
import asyncio
import pygame
import edge_tts
import time
import os

class VoiceAssistant:
    def __init__(self):
        # 1. Safe Audio Init (Prevents crash if no speakers found)
        try:
            pygame.mixer.init()
        except Exception as e:
            print(f"Warning: Audio device not found. {e}")
            
        self.last_said_time = 0
        self.voice_name = "fr-FR-HenriNeural" 
        self.is_speaking = False  # Lock to prevent overlapping audio

    def speak(self, text, force=False):
        current_time = time.time()
        
        # SPAM FILTER: Don't repeat advice more than once every 3 seconds
        if not force and (current_time - self.last_said_time < 3):
            return

        # OVERLAP FILTER: Don't speak if already speaking (unless forced)
        if self.is_speaking and not force:
            return

        self.last_said_time = current_time
        
        # 2. FIXED THREADING: Use Daemon Mode
        # Daemon threads automatically die when the main program closes.
        # Without this, your app hangs in the background forever.
        t = threading.Thread(target=self._run_tts_safe, args=(text,))
        t.daemon = True 
        t.start()

    def _run_tts_safe(self, text):
        # 3. FIXED ASYNC: Manual Event Loop
        # 'asyncio.run' crashes in complex apps. We must create a 
        # fresh, private loop for this specific thread.
        self.is_speaking = True
        try:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            loop.run_until_complete(self._generate_and_play(text))
            loop.close()
        except Exception as e:
            print(f"TTS Error: {e}")
        finally:
            self.is_speaking = False

    async def _generate_and_play(self, text):
        filename = f"temp_{int(time.time() * 1000)}.mp3"
        try:
            # Connect to Microsoft and download
            communicate = edge_tts.Communicate(text, self.voice_name)
            await communicate.save(filename)
            
            # Play audio safely
            if pygame.mixer.get_init():
                pygame.mixer.music.load(filename)
                pygame.mixer.music.play()
                
                while pygame.mixer.music.get_busy():
                    pygame.time.Clock().tick(10)
        except Exception as e:
            print(f"Playback Error: {e}")
        finally:
            # CLEANUP: Must unload before deleting or Windows locks the file
            try:
                pygame.mixer.music.unload()
                if os.path.exists(filename):
                    os.remove(filename)
            except:
                pass

# Initialize Voice
voice = VoiceAssistant()

c:\Users\theda\anaconda3\envs\HAR\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


class VoiceAssistant:
    def __init__(self):
        # Initialize the audio player
        pygame.mixer.init()
        self.last_said_time = 0
        
        # VOICE SELECTION
        # "en-US-ChristopherNeural" = Male Trainer (Deep, Calm)
        # "en-US-AriaNeural" = Female Trainer (Clear, Energetic)
        # "en-US-GuyNeural" = Generic Male
        self.voice_name = "en-US-AriaNeural" 

    def speak(self, text, force=False):
        current_time = time.time()
        
        # SPAM FILTER: Don't repeat advice more than once every 3 seconds
        if not force and (current_time - self.last_said_time < 3):
            return

        self.last_said_time = current_time
        
        # Run in a separate thread to prevent camera freeze
        threading.Thread(target=self._run_tts_async, args=(text,)).start()

    def _run_tts_async(self, text):
        # We need to run the async edge-tts code inside a standard thread
        try:
            asyncio.run(self._generate_and_play(text))
        except Exception as e:
            print(f"TTS Error: {e}")

    async def _generate_and_play(self, text):
        # 1. Generate a unique filename (to avoid file permission errors)
        filename = f"temp_voice_{int(time.time() * 1000)}.mp3"
        
        # 2. Connect to Microsoft and download the audio
        communicate = edge_tts.Communicate(text, self.voice_name)
        await communicate.save(filename)
        
        # 3. Play the audio using Pygame
        # We use a try/finally block to ensure we clean up the file afterwards
        try:
            pygame.mixer.music.load(filename)
            pygame.mixer.music.play()
            
            # Wait for audio to finish so we don't delete the file while playing
            while pygame.mixer.music.get_busy():
                pygame.time.Clock().tick(10)
                
        finally:
            # 4. Cleanup: Delete the temp file to save space
            pygame.mixer.music.unload()
            if os.path.exists(filename):
                try:
                    os.remove(filename)
                except:
                    pass

# Initialize Voice
voice = VoiceAssistant()

## Squat ##

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
from dataclasses import dataclass

# New Tasks API Imports
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# --- CONFIGURATION ---
model_path = 'C:\\sample_path\\pose_landmarker_lite.task' 
model_path="C:\\sample_path\\pose_landmarker_heavy.task"

# ==========================================
# 1. GEOMETRY & MATH UTILS
# ==========================================
class PoseUtils:
    @staticmethod
    def calculate_angle(a, b, c):
        """Calculate angle at joint b (a-b-c)"""
        a = np.array([a.x, a.y])
        b = np.array([b.x, b.y])
        c = np.array([c.x, c.y])

        radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
        angle = np.abs(radians * 180.0 / np.pi)

        if angle > 180.0:
            angle = 360 - angle
        return angle

# ==========================================
# 2. EXERCISE LOGIC
# ==========================================
@dataclass
class ExerciseState:
    angle: float
    stage: str
    reps: int
    feedback: str
    color: tuple

class SquatAnalyzer:
    def __init__(self):
        self.reps = 0
        self.stage = "UP"
        self.feedback = "Stand & Face Side"
        voice.speak("Stand and Face to the Side", force=False)
        self.color = (255, 255, 255)
        self.bottom_reached = False # Flag to ensure full ROM

    def analyze(self, landmarks):
        # 1. AUTO-DETECT SIDE
        # Check visibility of Left Hip (23) vs Right Hip (24)
        # We use the side that is "more visible" (usually lower z value or higher visibility score)
        # Simple heuristic: Use the side with landmarks closer to the camera.
        
        left_vis = landmarks[23].visibility if hasattr(landmarks[23], 'visibility') else 1.0
        right_vis = landmarks[24].visibility if hasattr(landmarks[24], 'visibility') else 1.0
        
        # Default to left, switch if right is significantly clearer
        side = "LEFT"
        if landmarks[24].visibility > landmarks[23].visibility:
            side = "RIGHT"

        # Map indices based on side
        if side == "LEFT":
            hip, knee, ankle, shoulder = landmarks[23], landmarks[25], landmarks[27], landmarks[11]
        else: # RIGHT
            hip, knee, ankle, shoulder = landmarks[24], landmarks[26], landmarks[28], landmarks[12]

        # 2. CALCULATE ANGLE
        angle = PoseUtils.calculate_angle(hip, knee, ankle)
        
        # 3. CHEAT DETECTION (Direction Agnostic)
        # Use abs() so it works facing Left or Right
        # If Shoulder X is too far ahead of Knee X (leaning forward)
        chest_lean = shoulder.x - knee.x
        
        # Adjust logic for side (if facing right vs left, sign changes)
        # Simplified: If the horizontal distance is massive, it's a lean.
        if abs(chest_lean) > 0.05: 
            self.feedback = "CHEST UP!"
            voice.speak("CHEST UP!", force=False)
            self.color = (0, 0, 255) # Red
        
        # 4. REP LOGIC (Hysteresis)
        else:
            # DOWN PHASE
            if angle < 90:
                self.stage = "DOWN"
                self.bottom_reached = True # Flag that we hit depth
                self.feedback = "NICE!"
                self.color = (0, 255, 0) # Green
            
            # UP PHASE (Count Rep Here)
            elif angle > 160:
                if self.stage == "DOWN" and self.bottom_reached:
                    self.reps += 1
                    voice.speak(str(self.reps), force=False)
                    self.bottom_reached = False # Reset flag
                
                self.stage = "UP"
                self.feedback = "SQUAT"
                self.color = (255, 255, 0) # Yellow

        return ExerciseState(angle, self.stage, self.reps, self.feedback, self.color)

# ==========================================
# 3. VISUALIZATION
# ==========================================
def draw_skeleton(image, landmarks):
    h, w, _ = image.shape
    # Simple connections for legs and torso
    connections = [
        (11, 12), (11, 23), (12, 24), (23, 24), # Torso
        (23, 25), (25, 27), (24, 26), (26, 28)  # Legs
    ]
    
    for start, end in connections:
        if start < len(landmarks) and end < len(landmarks):
            p1 = landmarks[start]
            p2 = landmarks[end]
            cv2.line(image, (int(p1.x*w), int(p1.y*h)), (int(p2.x*w), int(p2.y*h)), (255,255,255), 2)
            cv2.circle(image, (int(p1.x*w), int(p1.y*h)), 4, (0,255,255), -1)

# ==========================================
# 4. MAIN
# ==========================================
def run_app():
    # Setup MediaPipe
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        output_segmentation_masks=False,
        running_mode=vision.RunningMode.VIDEO 
    )

    analyzer = SquatAnalyzer()
    
    # Open Camera
    cap = cv2.VideoCapture(0)
    # Set to 720p for better detection
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            # --- KEY FIX: FLIP FIRST ---
            # Flip the frame horizontally immediately. 
            # Now "Left" on screen is "Left" in math.
            frame = cv2.flip(frame, 1)

            # Convert for MediaPipe
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            timestamp_ms = int(time.time() * 1000)
            
            # Detect
            detection_result = landmarker.detect_for_video(mp_image, timestamp_ms)

            # Process
            if detection_result.pose_landmarks:
                landmarks = detection_result.pose_landmarks[0]
                
                # Analyze
                state = analyzer.analyze(landmarks)
                
                # Draw
                draw_skeleton(frame, landmarks)
                
                # UI Box
                cv2.rectangle(frame, (0,0), (300, 100), (30,30,30), -1)
                cv2.rectangle(frame, (0,0), (300, 100), state.color, 2)
                
                # Text
                cv2.putText(frame, f"REPS: {state.reps}", (20, 40), cv2.FONT_HERSHEY_DUPLEX, 1, (255,255,255), 2)
                
                cv2.putText(frame, state.feedback, (20, 80), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, state.color, 2)

                # Angle Debug (Optional)
                cv2.putText(frame, f"{int(state.angle)} deg", (20, 130), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (150,150,150), 1)

            cv2.imshow('Fixed Squat Counter', frame)
            if cv2.waitKey(5) & 0xFF == ord('q'):
                voice.speak("Workout Ended. Great Job!", force=True)
                break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_app()

## Shoulder Raise ##

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import sys

# --- CONFIGURATION ---
MODEL_PATH = r"C:\sample_path\pose_landmarker_lite.task"
GHOST_PATH = r"C:\sample_path\lateral_raise_ref.mp4" 
#============
# 1. SETUP & HELPER FUNCTIONS
# ==========================================
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def calculate_angle(a, b, c):
    """Calculates angle ABC where B is the vertex."""
    a_arr = np.array([a.x, a.y])
    b_arr = np.array([b.x, b.y])
    c_arr = np.array([c.x, c.y])

    radians = np.arctan2(c_arr[1] - b_arr[1], c_arr[0] - b_arr[0]) - \
              np.arctan2(a_arr[1] - b_arr[1], a_arr[0] - b_arr[0])
    
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0: angle = 360 - angle
    return angle

def draw_futuristic_skeleton(image, landmarks, connections):
    """Draws a neon 'Cyberpunk' style skeleton."""
    h, w, _ = image.shape
    
    # Colors (BGR)
    NEON_CYAN = (255, 255, 0)      # Bones
    NEON_MAGENTA = (255, 0, 255)   # Joints
    WHITE = (255, 255, 255)

    # 1. Draw Bones (with Glow Effect)
    for start, end in connections:
        p1 = landmarks[start]
        p2 = landmarks[end]
        
        x1, y1 = int(p1.x * w), int(p1.y * h)
        x2, y2 = int(p2.x * w), int(p2.y * h)
        
        # Outer Glow (Thick, transparent-ish look)
        cv2.line(image, (x1, y1), (x2, y2), NEON_CYAN, 4) 
        # Inner Core (Thin, bright)
        cv2.line(image, (x1, y1), (x2, y2), WHITE, 1)

    # 2. Draw Joints (Tech Nodes)
    # We only care about upper body for this exercise
    joint_indices = [11, 12, 13, 14, 15, 16, 23, 24] 
    for idx in joint_indices:
        p = landmarks[idx]
        cx, cy = int(p.x * w), int(p.y * h)
        
        # Outer Ring
        cv2.circle(image, (cx, cy), 8, NEON_MAGENTA, 1)
        # Inner Dot
        cv2.circle(image, (cx, cy), 3, WHITE, -1)


# ==========================================
# IMPROVED MEDIAPIPE SETUP
# ==========================================
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=False,
    # 1. DETECTION: Lower this to 0.4 so it doesn't "lose" you easily
    min_pose_detection_confidence=0.2, 
    # 2. TRACKING: Keep this at 0.5 or higher to prevent jitter
    min_tracking_confidence=0.2
)
detector = vision.PoseLandmarker.create_from_options(options)
detector = vision.PoseLandmarker.create_from_options(options)

# Initialize Camera
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1960)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

# Initialize Ghost
ghost_cap = cv2.VideoCapture(GHOST_PATH)
has_ghost = ghost_cap.isOpened()

POSE_CONNECTIONS = [
    (11, 12), (11, 23), (12, 24), (23, 24), # Torso
    (12, 14), (14, 16), # Right Arm
    (11, 13), (13, 15)  # Left Arm
]

# ==========================================
# 2. EXERCISE VARIABLES
# ==========================================
reps = 0
stage = "DOWN"
feedback = "Start"
feedback_color = (200, 200, 200)

while True:
    ret, frame = cap.read()
    if not ret: break

    h, w, _ = frame.shape

    # Darken background slightly to make neon pop
    frame = cv2.addWeighted(frame, 0.8, np.zeros(frame.shape, frame.dtype), 0, -20)

    # --- GHOST LOGIC ---
    if has_ghost:
        g_ret, g_frame = ghost_cap.read()
        if not g_ret:
            ghost_cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            g_ret, g_frame = ghost_cap.read()
        
        if g_ret:
            g_frame = cv2.resize(g_frame, (w, h))
            # Lower opacity for ghost (hologram style)
            frame = cv2.addWeighted(frame, 0.8, g_frame, 0.2, 0)

    # --- DETECTION ---
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    detection_result = detector.detect(mp_image)

    if detection_result.pose_landmarks:
        landmarks = detection_result.pose_landmarks[0]
        
        # Draw Sci-Fi Skeleton
        draw_futuristic_skeleton(frame, landmarks, POSE_CONNECTIONS)

        # Logic Vars
        r_hip = landmarks[24]
        r_shoulder = landmarks[12]
        r_elbow = landmarks[14]
        r_wrist = landmarks[16]

        abduction_angle = calculate_angle(r_hip, r_shoulder, r_elbow)
        elbow_angle = calculate_angle(r_shoulder, r_elbow, r_wrist)

        # --- HYSTERESIS LOGIC (Anti-Flicker) ---
        # 1. DOWN STATE: Must drop VERY low (below 25) to reset
        if abduction_angle < 25:
            stage = "DOWN"
            feedback = "LIFT"
            feedback_color = (255, 255, 255)
        
        # 2. UP STATE: Must go HIGH (above 85) to count
        # This gap (25 to 85) prevents flickering
        if abduction_angle > 85 and stage == "DOWN":
            if elbow_angle > 120:
                stage = "UP"
                reps += 1
                
                voice.speak(str(reps), force=False)
                feedback = "PERFECT"
                feedback_color = (0, 255, 0) # Green
            else:
                feedback = "TOO BENT"
                voice.speak("TOO BENT", force=False)
                feedback_color = (0, 0, 255) # Red

        # --- UI OVERLAY ---
        # Draw Tech Box
        cv2.rectangle(frame, (0,0), (250, 90), (50, 50, 50), -1) # Dark grey background
        cv2.rectangle(frame, (0,0), (250, 90), (255, 255, 0), 2) # Cyan border
        
        # Stats
        cv2.putText(frame, f"REPS: {reps}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
        cv2.putText(frame, feedback, (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, feedback_color, 2)

        # Real-time Angle Data (Small text next to shoulder)
        sx, sy = int(r_shoulder.x * w), int(r_shoulder.y * h)
        cv2.putText(frame, f"{int(abduction_angle)} deg", (sx+20, sy), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

    cv2.imshow('Neon Trainer', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        voice.speak("Workout Ended. Great Job!", force=True)
        break
        

cap.release()
if has_ghost: ghost_cap.release()
cv2.destroyAllWindows()

## Plank ##


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import sys

import threading
import asyncio
import edge_tts
import pygame
import os
import time


# --- CONFIGURATION ---
# Use the "Full" model if you downloaded it, otherwise "Lite"
MODEL_PATH = r"C:\sample_path\pose_landmarker_lite.task"
# Optional: Use a video of someone holding a perfect plank
GHOST_PATH = r"C:\sample_path\plank_ref.mp4" 

# ==========================================
# 1. SETUP
# ==========================================
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def calculate_angle(a, b, c):
    """Calculates angle ABC."""
    a_arr = np.array([a.x, a.y])
    b_arr = np.array([b.x, b.y])
    c_arr = np.array([c.x, c.y])

    radians = np.arctan2(c_arr[1] - b_arr[1], c_arr[0] - b_arr[0]) - \
              np.arctan2(a_arr[1] - b_arr[1], a_arr[0] - b_arr[0])
    
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0: angle = 360 - angle
    return angle

def draw_neon_skeleton(image, landmarks):
    h, w, _ = image.shape
    # Neon Colors
    CYAN = (255, 255, 0)
    MAGENTA = (255, 0, 255)
    WHITE = (255, 255, 255)
    
    # Plank Connections (Right Side Focus)
    connections = [
        (12, 14), (14, 16), # Right Arm
        (12, 24), (24, 26), (26, 28), # Body Line (Shoulder-Hip-Knee-Ankle)
        (11, 12), (11, 23), (23, 24) # Torso Box
    ]

    for start, end in connections:
        p1, p2 = landmarks[start], landmarks[end]
        x1, y1 = int(p1.x * w), int(p1.y * h)
        x2, y2 = int(p2.x * w), int(p2.y * h)
        cv2.line(image, (x1, y1), (x2, y2), CYAN, 4) # Glow
        cv2.line(image, (x1, y1), (x2, y2), WHITE, 1) # Core

    # Draw Hip Joint (The Critical Point)
    hip = landmarks[24]
    hx, hy = int(hip.x * w), int(hip.y * h)
    cv2.circle(image, (hx, hy), 8, MAGENTA, 1)
    cv2.circle(image, (hx, hy), 4, WHITE, -1)

# Initialize Model
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=False,
    min_pose_detection_confidence=0.3,
    min_tracking_confidence=0.3
)
detector = vision.PoseLandmarker.create_from_options(options)

# Camera & Ghost
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1960)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

ghost_cap = cv2.VideoCapture(GHOST_PATH)
has_ghost = ghost_cap.isOpened()

# ==========================================
# 2. PLANK VARIABLES
# ==========================================
good_frames = 0
fps = 30 # Assume 30 FPS for timer
timer_seconds = 0

feedback = "GET INTO POSITION"
feedback_color = (200, 200, 200)

print("✅ Plank Tracker Started.")
voice.speak("Plank Tracker Started. Get into position.", force=False)

while True:
    ret, frame = cap.read()
    if not ret: break
    h, w, _ = frame.shape
    
    # Dark Mode for Neon
    frame = cv2.addWeighted(frame, 0.7, np.zeros(frame.shape, frame.dtype), 0, -30)

    # --- GHOST OVERLAY ---
    if has_ghost:
        g_ret, g_frame = ghost_cap.read()
        if not g_ret:
            ghost_cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            g_ret, g_frame = ghost_cap.read()
        if g_ret:
            g_frame = cv2.resize(g_frame, (w, h))
            frame = cv2.addWeighted(frame, 0.8, g_frame, 0.2, 0)

    # --- DETECTION ---
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    detection_result = detector.detect(mp_image)

    if detection_result.pose_landmarks:
        landmarks = detection_result.pose_landmarks[0]
        draw_neon_skeleton(frame, landmarks)

        # 1. Get Key Joints (Right Side)
        shoulder = landmarks[12]
        hip = landmarks[24]
        ankle = landmarks[28]

        # 2. Calculate Alignment Angle
        hip_angle = calculate_angle(shoulder, hip, ankle)

        ## 3. PLANK LOGIC (STABILIZED)

        # A. Orientation Check: Are you standing up?
        # increased threshold to 0.2 to be more forgiving for different camera angles
        if abs(shoulder.y - hip.y) > 0.2: 
            feedback = "GET HORIZONTAL"
            feedback_color = (255, 255, 255)
        
        else:
            # B. Calculate the "Ideal Line"
            # If your body was a straight line, this is exactly where your hip SHOULD be.
            midpoint_y = (shoulder.y + ankle.y) / 2
            
            # Calculate the Error: How far is your actual hip from that ideal point?
            # Negative = Hip is higher than line (Piking)
            # Positive = Hip is lower than line (Sagging)
            error = hip.y - midpoint_y

            # C. The "Buffer Zone" Logic
            
            # CASE 1: Perfect Form (Angle is straight)
            if hip_angle > 160:
                feedback = "HOLD IT!"
                feedback_color = (0, 255, 0) # Green
                good_frames += 1
                timer_seconds = int(good_frames / fps)

            # CASE 2: Bad Angle, but "Close Enough" (The Anti-Flicker Fix)
            # If your angle is slightly off, but your hip is within 0.03 of the line,
            # we count it as good to prevent the UI from jumping around.
            elif abs(error) < 0.03: 
                feedback = "HOLD IT!"
                feedback_color = (0, 255, 0)
                good_frames += 1
                timer_seconds = int(good_frames / fps)

            # CASE 3: Significant Error (Real Correction needed)
            else:
                if error < -0.03: 
                    # You are WAY above the line
                    feedback = "LOWER HIPS!" 
                    voice.speak("LOWER HIPS", force=False)
                    feedback_color = (0, 0, 255) # Red
                elif error > 0.03:
                    # You are WAY below the line
                    feedback = "RAISE HIPS!"
                    voice.speak("RAISE HIPS", force=False)
                    feedback_color = (0, 0, 255) # Red
        # --- UI DISPLAY ---
        # 1. Timer Box
        cv2.rectangle(frame, (w-220, 0), (w, 100), (30, 30, 30), -1)
        cv2.rectangle(frame, (w-220, 0), (w, 100), (255, 255, 0), 2)
        
        cv2.putText(frame, "TIME HELD", (w-200, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
        
        # Format Time as MM:SS
        mins, secs = divmod(timer_seconds, 60)
        time_str = f"{mins:02}:{secs:02}"
        cv2.putText(frame, time_str, (w-200, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)

        # 2. Feedback Text (Center)
        text_size = cv2.getTextSize(feedback, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)[0]
        text_x = (w - text_size[0]) // 2
        cv2.putText(frame, feedback, (text_x, h - 50), cv2.FONT_HERSHEY_SIMPLEX, 1, feedback_color, 2)
        
        # 3. Angle Debug
        cv2.putText(frame, f"Angle: {int(hip_angle)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

    else:
        cv2.putText(frame, "NO BODY DETECTED", (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow('Plank Trainer', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        voice.speak("Workout Ended. Great Job!", force=True)
        break

cap.release()
if has_ghost: ghost_cap.release()
cv2.destroyAllWindows()

✅ Plank Tracker Started.


## BICEP CURLS ##

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import threading # Required to stop the camera from freezing
import pyttsx3   # The Text-to-Speech library
from dataclasses import dataclass
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# --- CONFIGURATION ---
model_path = 'C:\\sample_path\\pose_landmarker_lite.task' 

# ==========================================
# 1. VOICE ASSISTANT (The New Part)
# ==========================================
import threading
import asyncio
import edge_tts
import pygame
import os
import time


# Initialize Voice
voice = VoiceAssistant()

# ==========================================
# 2. GEOMETRY UTILS
# ==========================================
def calculate_angle(a, b, c):
    a = np.array([a.x, a.y])
    b = np.array([b.x, b.y])
    c = np.array([c.x, c.y])
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0: angle = 360 - angle
    return angle

# ==========================================
# 3. EXERCISE LOGIC
# ==========================================
@dataclass
class CurlState:
    angle: float
    reps: int
    feedback: str
    color: tuple
    swing_score: int

class BicepCurlAnalyzer:
    def __init__(self):
        self.reps = 0
        self.stage = "DOWN"
        self.extended_reached = True
        self.last_rep_time = 0
        self.display_duration = 1.5

    def analyze(self, landmarks):
        # AUTO-DETECT SIDE
        left_vis = landmarks[11].visibility
        right_vis = landmarks[12].visibility
        
        if right_vis > left_vis:
            shoulder, elbow, wrist, hip = landmarks[12], landmarks[14], landmarks[16], landmarks[24]
        else:
            shoulder, elbow, wrist, hip = landmarks[11], landmarks[13], landmarks[15], landmarks[23]

        curl_angle = calculate_angle(shoulder, elbow, wrist)
        swing_angle = calculate_angle(hip, shoulder, elbow)

        current_feedback = "Extend Arm"
        current_color = (255, 255, 255)
        
        # --- LOGIC ---
        if curl_angle > 160:
            self.stage = "DOWN"
            self.extended_reached = True
            current_feedback = "Curl Up"
        
        elif curl_angle < 50:
            if self.stage == "DOWN" and self.extended_reached:
                self.stage = "UP"
                self.reps += 1
                self.extended_reached = False
                self.last_rep_time = time.time()
                
                # --- VOICE TRIGGER: REP COUNT ---
                voice.speak(str(self.reps), force=False) 
                
            elif not self.extended_reached:
                current_feedback = "Full Extension!"
                current_color = (0, 255, 255)
                # --- VOICE TRIGGER: ADVICE ---
                voice.speak("Go all the way down", force=False)

        # --- FEEDBACK PRIORITY ---
        final_feedback = current_feedback
        final_color = current_color

        # 1. SAFETY (Highest Priority)
        if swing_angle > 25:
            final_feedback = "GLUE ELBOW!"
            final_color = (0, 0, 255)
            # --- VOICE TRIGGER: WARNING ---
            voice.speak("Don't swing your elbow", force=False)
        
        # 2. SUCCESS (Medium Priority)
        elif (time.time() - self.last_rep_time) < self.display_duration:
            final_feedback = "PERFECT! ✅"
            final_color = (0, 255, 0)
            # (We already spoke the number, no need to speak "Perfect" too)

        return CurlState(curl_angle, self.reps, final_feedback, final_color, int(swing_angle))

# ==========================================
# 4. VISUALS & RUNNER
# ==========================================
def draw_neon_overlay(image, landmarks, state):
    h, w, _ = image.shape
    # Draw Skeleton
    connections = [(11, 12), (11, 23), (12, 24), (23, 24), (12, 14), (14, 16), (11, 13), (13, 15)]
    for s, e in connections:
        if s < len(landmarks) and e < len(landmarks):
            p1, p2 = landmarks[s], landmarks[e]
            cv2.line(image, (int(p1.x*w), int(p1.y*h)), (int(p2.x*w), int(p2.y*h)), (255, 255, 0), 4)

    # UI Box
    cv2.rectangle(image, (0,0), (350, 100), (20, 20, 20), -1)
    cv2.rectangle(image, (0,0), (350, 100), state.color, 2)
    cv2.putText(image, f"REPS: {state.reps}", (15, 40), cv2.FONT_HERSHEY_DUPLEX, 1, (255, 255, 255), 2)
    cv2.putText(image, state.feedback, (15, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, state.color, 2)
    cv2.putText(image, f"Swing: {state.swing_score}deg", (200, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)

def run_curls_with_voice():
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        output_segmentation_masks=False,
        running_mode=vision.RunningMode.VIDEO 
    )

    analyzer = BicepCurlAnalyzer()
    
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    # Announce start
    voice.speak("Starting Bicep Curls. Get ready.", force=True)

    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            frame = cv2.flip(frame, 1)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            timestamp = int(time.time() * 1000)
            
            detection_result = landmarker.detect_for_video(mp_image, timestamp)

            if detection_result.pose_landmarks:
                landmarks = detection_result.pose_landmarks[0]
                state = analyzer.analyze(landmarks)
                draw_neon_overlay(frame, landmarks, state)

            cv2.imshow('Talking AI Trainer', frame)
            if cv2.waitKey(5) & 0xFF == ord('q'):
                voice.speak("Workout Ended. Great Job!", force=True)
                break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_curls_with_voice()

Straight leg raise

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import threading
import asyncio
import edge_tts
import pygame
import os
from dataclasses import dataclass
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# --- CONFIGURATION ---
model_path = 'C:\\sample_path\\pose_landmarker_lite.task' 



# ==========================================
# 2. MATH UTILS
# ==========================================
def calculate_angle(a, b, c):
    a = np.array([a.x, a.y])
    b = np.array([b.x, b.y])
    c = np.array([c.x, c.y])
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0: angle = 360 - angle
    return angle

# ==========================================
# 3. REHAB LOGIC: Straight Leg Raise (SLR)
# ==========================================
@dataclass
class SLRState:
    hip_angle: float
    knee_angle: float
    reps: int
    feedback: str
    color: tuple
    rep_quality: bool # Did they maintain form efficiently?

class SLRAnalyzer:
    def __init__(self):
        self.reps = 0
        self.stage = "DOWN"
        self.rep_quality = True # Starts true, becomes False if you cheat
        self.feedback = "Lie down flat"
        self.last_rep_time = 0

    def analyze(self, landmarks):
        # 1. AUTO-DETECT SIDE (which leg is moving?)
        # For SLR, we assume the user is lying side-profile.
        # We track the leg that is CLOSER to the camera (higher visibility)
        left_vis = landmarks[23].visibility
        right_vis = landmarks[24].visibility
        
        if right_vis > left_vis:
            # Right Side facing camera
            hip, knee, ankle, shoulder = landmarks[24], landmarks[26], landmarks[28], landmarks[12]
        else:
            # Left Side facing camera
            hip, knee, ankle, shoulder = landmarks[23], landmarks[25], landmarks[27], landmarks[11]

        # 2. CALCULATE ANGLES
        # Knee Angle: Must stay ~180
        knee_angle = calculate_angle(hip, knee, ankle)
        # Hip Angle: 180 (flat) -> 135 (raised)
        hip_angle = calculate_angle(shoulder, hip, knee)
        
        # 3. SAFETY CHECKS (The Rehab "Strictness")
        # If knee bends significantly (< 165), the rep is ruined.
        if knee_angle < 130:
            self.rep_quality = False
            self.feedback = "KNEE BENT! LOCK IT!"
            return SLRState(hip_angle, knee_angle, self.reps, self.feedback, (255, 0, 4), False)

        # 4. REP LOGIC
        # Leg is DOWN (Flat on floor)
        if hip_angle > 130:
            if self.stage == "UP":
                self.stage = "DOWN"
                # Only count if the rep quality was maintained the WHOLE time
                if self.rep_quality:
                    self.reps += 1
                    self.last_rep_time = time.time()
                    voice.speak(f"Good. That is {self.reps}.", force=False)
                else:
                    voice.speak("Rep failed. Keep knee straight.", force=True)
            
            # Reset quality for the next attempt
            self.rep_quality = True
            self.feedback = "Raise Leg Slowly"
            return SLRState(hip_angle, knee_angle, self.reps, self.feedback, (255, 255, 255), True)

        # Leg is UP (Active range)
        elif hip_angle < 140:
            self.stage = "UP"
            
            if self.rep_quality:
                self.feedback = "HOLD..."
                color = (0, 255, 0)
            else:
                self.feedback = "Failed Rep"
                color = (0, 0, 255)
                
            return SLRState(hip_angle, knee_angle, self.reps, self.feedback, color, self.rep_quality)

        # In-Between (Moving)
        else:
            if self.rep_quality:
                self.feedback = "Keep Knee Locked..."
                color = (255, 255, 0)
            else:
                self.feedback = "Straighten Knee!"
                color = (0, 0, 255)
            
            return SLRState(hip_angle, knee_angle, self.reps, self.feedback, color, self.rep_quality)

# ==========================================
# 4. REHAB VISUALIZATION
# ==========================================
def draw_rehab_overlay(image, landmarks, state):
    h, w, _ = image.shape
    
    # Visual: Draw a "Cast" or "Brace" on the leg
    # If quality is good, draw the leg in Green. If bad, Red.
    color = (0, 255, 0) if state.rep_quality else (0, 0, 255)
    
    # We need the indices again (simplified for drawing, just grabs the visible ones)
    # This is a quick hack to draw whichever side is visible
    if landmarks[24].visibility > landmarks[23].visibility:
        pts = [24, 26, 28] # Right leg
    else:
        pts = [23, 25, 27] # Left leg
        
    # Draw thick "bone" lines
    p_hip = landmarks[pts[0]]
    p_knee = landmarks[pts[1]]
    p_ankle = landmarks[pts[2]]
    
    cv2.line(image, (int(p_hip.x*w), int(p_hip.y*h)), (int(p_knee.x*w), int(p_knee.y*h)), color, 6)
    cv2.line(image, (int(p_knee.x*w), int(p_knee.y*h)), (int(p_ankle.x*w), int(p_ankle.y*h)), color, 6)
    
    # Draw Knee Joint Angle Bubble
    k_x, k_y = int(p_knee.x*w), int(p_knee.y*h)
    cv2.circle(image, (k_x, k_y), 10, (255,255,255), -1)
    
    # UI Text
    cv2.rectangle(image, (0,0), (400, 120), (245, 245, 245), -1)
    
    # Feedback
    cv2.putText(image, state.feedback, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)
    
    # Knee Metric
    knee_text_color = (0, 100, 0) if state.knee_angle > 165 else (0, 0, 255)
    cv2.putText(image, f"Knee Straightness: {int(state.knee_angle)} deg", (20, 90), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, knee_text_color, 2)
                
    # Reps
    cv2.putText(image, f"Good Reps: {state.reps}", (20, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

# ==========================================
# 5. RUNNER
# ==========================================
def run_rehab_slr():
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        output_segmentation_masks=False,
        running_mode=vision.RunningMode.VIDEO 
    )

    analyzer = SLRAnalyzer()
    
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    voice.speak("Starting Leg Raises. Lie on your side. Keep your knee perfectly straight.", force=True)

    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            frame = cv2.flip(frame, 1)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            timestamp = int(time.time() * 1000)
            
            detection_result = landmarker.detect_for_video(mp_image, timestamp)

            if detection_result.pose_landmarks:
                landmarks = detection_result.pose_landmarks[0]
                state = analyzer.analyze(landmarks)
                draw_rehab_overlay(frame, landmarks, state)

            cv2.imshow('Rehab AI: Straight Leg Raise', frame)
            if cv2.waitKey(5) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_rehab_slr()

Starting glute curl

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import threading
import asyncio
import edge_tts
import pygame
import os
from dataclasses import dataclass
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# --- CONFIGURATION ---
model_path = 'C:\\sample_path\\pose_landmarker_heavy.task' 

# ==========================================
# 1. CONSTANTS (Manually Defined)
# ==========================================
# Standard MediaPipe Pose Connections (Reference)
# We define them here to avoid importing the legacy 'mp.solutions'
POSE_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15), # Arms
    (12, 14), (14, 16),
    (11, 23), (12, 24), # Torso
    (23, 24),
    (23, 25), (24, 26), # Thighs
    (25, 27), (26, 28)  # Calves
]

# ==========================================
# 2. VOICE ASSISTANT
# ==========================================


# ==========================================
# 3. MATH & STABILIZERS
# ==========================================
def calculate_angle(a, b, c):
    a = np.array([a.x, a.y])
    b = np.array([b.x, b.y])
    c = np.array([c.x, c.y])
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0: angle = 360 - angle
    return angle

class Stabilizer:
    def __init__(self, alpha=0.3):
        self.alpha = alpha
        self.value = 175.0 # Assume straight leg start

    def update(self, new_value):
        self.value = (self.alpha * new_value) + ((1 - self.alpha) * self.value)
        return self.value

# ==========================================
# 4. HAMSTRING CURL ANALYZER
# ==========================================
@dataclass
class CurlState:
    r_angle: float
    l_angle: float
    active_leg: str
    reps: int
    feedback: str
    r_vis: float
    l_vis: float

class CurlAnalyzer:
    def __init__(self):
        self.r_stab = Stabilizer(alpha=0.3)
        self.l_stab = Stabilizer(alpha=0.3)
        self.reps = 0
        self.stage = "DOWN"
        self.active_leg = "NONE"
        self.feedback = "Stand Straight"

    def analyze(self, landmarks):
        # 1. GET COORDINATES
        r_hip, r_knee, r_ankle = landmarks[24], landmarks[26], landmarks[28]
        l_hip, l_knee, l_ankle = landmarks[23], landmarks[25], landmarks[27]

        # 2. CALCULATE ANGLES
        raw_r = calculate_angle(r_hip, r_knee, r_ankle)
        raw_l = calculate_angle(l_hip, l_knee, l_ankle)
        
        # 3. SMOOTHING
        r_angle = self.r_stab.update(raw_r)
        l_angle = self.l_stab.update(raw_l)

        # 4. DETECT ACTIVE LEG
        # If a leg bends significantly (< 155), it becomes the active curling leg
        if r_angle < 155 and l_angle > 165:
            self.active_leg = "RIGHT"
        elif l_angle < 155 and r_angle > 165:
            self.active_leg = "LEFT"
        elif r_angle > 165 and l_angle > 165:
            pass # Maintain state during reset
            
        # 5. REP LOGIC
        if self.active_leg == "RIGHT":
            angle = r_angle
            stance_angle = l_angle
        elif self.active_leg == "LEFT":
            angle = l_angle
            stance_angle = r_angle
        else:
            angle = 180
            stance_angle = 180

        # Posture Check
        if self.active_leg != "NONE" and stance_angle < 155:
            self.feedback = "Don't bend standing leg!"
            return CurlState(r_angle, l_angle, self.active_leg, self.reps, self.feedback, r_knee.visibility, l_knee.visibility)

        # Counting
        if self.active_leg != "NONE":
            # Target: Curl to 90 degrees or less
            if angle < 95: 
                self.stage = "CURL"
                self.feedback = "HOLD!"
            
            # Reset: Return to straight (> 165)
            if angle > 165 and self.stage == "CURL":
                self.stage = "DOWN"
                self.reps += 1
                self.feedback = "Good."
                voice.speak(f"{self.active_leg} Rep {self.reps}", force=True)

            if self.stage == "DOWN":
                self.feedback = "Curl Heel Up"
        else:
            self.feedback = "Start Curling"

        return CurlState(r_angle, l_angle, self.active_leg, self.reps, self.feedback, r_knee.visibility, l_knee.visibility)

# ==========================================
# 5. PURE DRAWING FUNCTION
# ==========================================
def draw_curl_overlay(image, landmarks, state):
    h, w, _ = image.shape
    
    # 1. DRAW GHOST SKELETON (Using manual POSE_CONNECTIONS)
    # We iterate over our manually defined list, no legacy API needed.
    for start_idx, end_idx in POSE_CONNECTIONS:
        # Skip the legs in the ghost drawing (we draw them colored later)
        if start_idx in [23,24,25,26,27,28] and end_idx in [23,24,25,26,27,28]:
            continue
            
        p1 = landmarks[start_idx]
        p2 = landmarks[end_idx]
        
        if p1.visibility > 0.5 and p2.visibility > 0.5:
            cv2.line(image, (int(p1.x*w), int(p1.y*h)), (int(p2.x*w), int(p2.y*h)), (255, 255, 0), 2)
            cv2.circle(image, (int(p1.x*w), int(p1.y*h)), 2, (255, 0, 255), 8)

    # 2. DRAW LEFT LEG (Color Coded)
    l_color = (50, 50, 200) # Dark Red (Inactive)
    l_thick = 4
    if state.active_leg == "LEFT":
        l_color = (0, 255, 0) if state.l_angle < 95 else (0, 255, 255)
        l_thick = 8

    if state.l_vis > 0.5:
        p1, p2, p3 = landmarks[23], landmarks[25], landmarks[27]
        cv2.line(image, (int(p1.x*w), int(p1.y*h)), (int(p2.x*w), int(p2.y*h)), l_color, l_thick)
        cv2.line(image, (int(p2.x*w), int(p2.y*h)), (int(p3.x*w), int(p3.y*h)), l_color, l_thick)

    # 3. DRAW RIGHT LEG (Color Coded)
    r_color = (200, 50, 50) # Dark Blue (Inactive)
    r_thick = 4
    if state.active_leg == "RIGHT":
        r_color = (0, 255, 0) if state.r_angle < 95 else (0, 255, 255)
        r_thick = 8

    if state.r_vis > 0.5:
        p1, p2, p3 = landmarks[24], landmarks[26], landmarks[28]
        cv2.line(image, (int(p1.x*w), int(p1.y*h)), (int(p2.x*w), int(p2.y*h)), r_color, r_thick)
        cv2.line(image, (int(p2.x*w), int(p2.y*h)), (int(p3.x*w), int(p3.y*h)), r_color, r_thick)

    # 4. UI DASHBOARD
    cv2.rectangle(image, (0,0), (350, 150), (30, 30, 30), -1)
    
    cv2.putText(image, f"ACTIVE: {state.active_leg}", (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 2)
    feedback_color = (0, 255, 0) if "Good" in state.feedback else (0, 255, 255)
    cv2.putText(image, state.feedback, (15, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, feedback_color, 2)
    cv2.putText(image, f"REPS: {state.reps}", (15, 130), cv2.FONT_HERSHEY_DUPLEX, 1.2, (255, 255, 255), 2)

# ==========================================
# 6. RUNNER
# ==========================================
def run_standing_curl():
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        output_segmentation_masks=False,
        running_mode=vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    analyzer = CurlAnalyzer()
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    voice.speak("System Ready. Stand Sideways.", force=True)

    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            frame = cv2.flip(frame, 1)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            timestamp = int(time.time() * 1000)
            
            detection_result = landmarker.detect_for_video(mp_image, timestamp)

            if detection_result.pose_landmarks:
                landmarks = detection_result.pose_landmarks[0]
                state = analyzer.analyze(landmarks)
                draw_curl_overlay(frame, landmarks, state)

            cv2.imshow('Rehab AI: Standing Curl', frame)
            if cv2.waitKey(5) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    try: pygame.mixer.quit()
    except: pass

if __name__ == "__main__":
    run_standing_curl()